In [ ]:
# ==============================================================================
# Convert Raster Pixels to Points and Perform IDW Interpolation
#
# Author : Jintu Moni Bhuyan
# ==============================================================================

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.transform import xy, from_origin
from rasterio.features import geometry_mask
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

# ==============================================================================
# INPUT FILES
# ==============================================================================

input_raster = r"D:\NERDRR\July 2026\Nagaland_Jorhat\Assam\Giovani\Jorhat\Spatial\18 July\GIOVANNI-g4.accumulate.GPM_3IMERGHHL_07_precipitation.20260718-20260718.93E_26N_94E_27N.tif"

district_shp = r"D:\NERDRR\July 2026\Nagaland_Jorhat\Assam\Jorhat\Jorhat.shp"

output_raster = r"D:\NERDRR\July 2026\Nagaland_Jorhat\Assam\Giovani\Jorhat\Spatial\18 July\GIOVANNI-g4.accumulate.GPM_3IMERGHHL_07_precipitation.20260718-20260718_Interpolated_10m.tif"

# ==============================================================================
# PARAMETERS
# ==============================================================================

cell_size = 10      # metres
power = 2           # IDW power
k = 12              # nearest neighbours

# ==============================================================================
# READ SHAPEFILE
# ==============================================================================

boundary = gpd.read_file(district_shp)

print("\nBoundary CRS :", boundary.crs)

# ==============================================================================
# READ RASTER
# ==============================================================================

with rasterio.open(input_raster) as src:

    print("Raster CRS   :", src.crs)

    # -------------------------------------------------------------------------
    # Match CRS
    # -------------------------------------------------------------------------

    if boundary.crs != src.crs:
        boundary = boundary.to_crs(src.crs)
        print("Boundary reprojected to raster CRS.")

    # -------------------------------------------------------------------------
    # Clip Raster
    # -------------------------------------------------------------------------

    clipped, transform = mask(
        src,
        boundary.geometry,
        crop=True
    )

    image = clipped[0].astype(float)

    nodata = src.nodata

    if nodata is not None:
        image[image == nodata] = np.nan

    raster_crs = src.crs

# ==============================================================================
# Raster Pixels → Points
# ==============================================================================

rows, cols = np.where(~np.isnan(image))

xs = []
ys = []
vals = []

for r, c in zip(rows, cols):

    x, y = xy(transform, r, c, offset="center")

    xs.append(x)
    ys.append(y)
    vals.append(image[r, c])

points = gpd.GeoDataFrame(
    {"value": vals},
    geometry=gpd.points_from_xy(xs, ys),
    crs=raster_crs
)

print("Total Points :", len(points))

# ==============================================================================
# Reproject to UTM (Projected CRS)
# ==============================================================================

if points.crs.is_geographic:

    centroid = boundary.unary_union.centroid

    zone = int((centroid.x + 180) / 6) + 1

    epsg = 32600 + zone

    projected_crs = f"EPSG:{epsg}"

    print("Projected CRS :", projected_crs)

    boundary = boundary.to_crs(projected_crs)
    points = points.to_crs(projected_crs)

else:

    projected_crs = points.crs

# ==============================================================================
# Create 10 m Grid
# ==============================================================================

xmin, ymin, xmax, ymax = boundary.total_bounds

xgrid = np.arange(xmin, xmax + cell_size, cell_size)
ygrid = np.arange(ymin, ymax + cell_size, cell_size)

xx, yy = np.meshgrid(xgrid, ygrid)

# ==============================================================================
# IDW Interpolation
# ==============================================================================

coordinates = np.column_stack((points.geometry.x, points.geometry.y))

tree = cKDTree(coordinates)

k = min(k, len(points))

distances, indices = tree.query(
    np.column_stack((xx.ravel(), yy.ravel())),
    k=k
)

distances[distances == 0] = 1e-10

weights = 1 / distances**power

values = np.array(points["value"])

interpolated = np.sum(weights * values[indices], axis=1) / np.sum(weights, axis=1)

grid = interpolated.reshape(xx.shape)

# ==============================================================================
# Mask Outside District
# ==============================================================================

transform_out = from_origin(
    xmin,
    ymax,
    cell_size,
    cell_size
)

mask_array = geometry_mask(
    boundary.geometry,
    transform=transform_out,
    invert=True,
    out_shape=grid.shape
)

grid[~mask_array] = np.nan

# ==============================================================================
# Save Raster
# ==============================================================================

with rasterio.open(
    output_raster,
    "w",
    driver="GTiff",
    height=grid.shape[0],
    width=grid.shape[1],
    count=1,
    dtype="float32",
    crs=projected_crs,
    transform=transform_out,
    nodata=np.nan,
    compress="lzw"
) as dst:

    dst.write(grid.astype("float32"), 1)

print("\nInterpolated raster saved:")
print(output_raster)

# ==============================================================================
# Plot
# ==============================================================================

fig, ax = plt.subplots(figsize=(10, 8))

extent = (
    xmin,
    xmax,
    ymin,
    ymax
)

img = ax.imshow(
    grid,
    extent=extent,
    origin="upper",
    cmap="turbo"
)

boundary.boundary.plot(
    ax=ax,
    color="black",
    linewidth=1.5
)

cbar = plt.colorbar(img, ax=ax, shrink=0.75)
cbar.set_label("Rainfall (mm)", fontsize=11)

ax.set_title("IDW Interpolated Rainfall (10 m)", fontsize=14)
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

plt.tight_layout()
plt.show()


Boundary CRS : EPSG:32646
Raster CRS   : EPSG:4326
Boundary reprojected to raster CRS.
Total Points : 29
Projected CRS : EPSG:32646


C:\Users\Standard\AppData\Local\Temp\ipykernel_22344\1612737016.py:109: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = boundary.unary_union.centroid
